# LLM Evaluation: BLEU, ROUGE, LLM-as-Judge

Evaluating generative models is fundamentally harder than evaluating discriminative classifiers — there's no single right answer. This note implements BLEU and ROUGE from scratch, demonstrates LLM-as-judge position bias, and covers the eval stack used in production.

## What Interviewers Test
- Why automated metrics (BLEU, ROUGE) are insufficient and what they miss
- LLM-as-judge: how to use it and its failure modes (position bias, verbosity bias)
- Hallucination measurement: grounding-based and model-based approaches
- Eval set construction: golden sets, contamination, and coverage
- The full eval stack: automated + human + online signals

In [ ]:
import numpy as np
from collections import Counter
import math
np.random.seed(42)

# ===== BLEU Score from Scratch =====
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def modified_precision(candidate, references, n):
    """
    BLEU modified precision for n-grams.
    Clips n-gram counts to max count in any reference (prevents gaming with repetition).
    """
    candidate_ngrams = Counter(ngrams(candidate, n))
    max_ref_counts = Counter()
    for ref in references:
        ref_ngrams = Counter(ngrams(ref, n))
        for gram in ref_ngrams:
            max_ref_counts[gram] = max(max_ref_counts[gram], ref_ngrams[gram])
    
    clipped = {gram: min(count, max_ref_counts[gram])
               for gram, count in candidate_ngrams.items()}
    return sum(clipped.values()), sum(candidate_ngrams.values())

def bleu(candidate_tokens, references_tokens, max_n=4, weights=None):
    if weights is None:
        weights = [1.0/max_n] * max_n
    
    # Brevity penalty
    c = len(candidate_tokens)
    r = min((len(ref) for ref in references_tokens), key=lambda x: abs(x - c))
    bp = 1.0 if c >= r else math.exp(1 - r/c)
    
    log_sum = 0.0
    for n, w in enumerate(weights, 1):
        num, denom = modified_precision(candidate_tokens, references_tokens, n)
        if num == 0 or denom == 0:
            return 0.0
        log_sum += w * math.log(num / denom)
    
    return bp * math.exp(log_sum)

# Example
ref1 = "the quick brown fox jumps over the lazy dog".split()
ref2 = "a fast brown fox leaps over a lazy dog".split()
hyp_good = "the quick brown fox jumps over the lazy dog".split()
hyp_ok   = "a brown fox jumped over the lazy dog".split()
hyp_bad  = "the cat sat on the mat".split()

for label, hyp in [("Perfect", hyp_good), ("Good", hyp_ok), ("Bad", hyp_bad)]:
    score = bleu(hyp, [ref1, ref2])
    print(f"BLEU ({label:7s}): {score:.4f}")


In [ ]:
# ===== ROUGE-N and ROUGE-L =====
def rouge_n(candidate, reference, n):
    """Recall-oriented: how many reference n-grams appear in candidate?"""
    cand_ngrams = Counter(ngrams(candidate, n))
    ref_ngrams  = Counter(ngrams(reference, n))
    
    overlap = sum(min(cand_ngrams[g], ref_ngrams[g]) for g in ref_ngrams)
    ref_total = sum(ref_ngrams.values())
    precision = overlap / (sum(cand_ngrams.values()) + 1e-8)
    recall    = overlap / (ref_total + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return {'precision': precision, 'recall': recall, 'f1': f1}

def rouge_l(candidate, reference):
    """Longest Common Subsequence F1."""
    m, n = len(candidate), len(reference)
    dp = [[0] * (n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if candidate[i-1] == reference[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    lcs = dp[m][n]
    p = lcs / (m + 1e-8)
    r = lcs / (n + 1e-8)
    f1 = 2 * p * r / (p + r + 1e-8)
    return {'lcs': lcs, 'precision': p, 'recall': r, 'f1': f1}

ref = "machine learning models require large datasets for training".split()
hyp = "machine learning needs large amounts of data for training".split()

r1 = rouge_n(hyp, ref, n=1)
r2 = rouge_n(hyp, ref, n=2)
rl = rouge_l(hyp, ref)
print(f"ROUGE-1 F1: {r1['f1']:.4f}  (precision={r1['precision']:.4f}, recall={r1['recall']:.4f})")
print(f"ROUGE-2 F1: {r2['f1']:.4f}  (precision={r2['precision']:.4f}, recall={r2['recall']:.4f})")
print(f"ROUGE-L F1: {rl['f1']:.4f}  (LCS length={rl['lcs']})")
print()
print("BLEU/ROUGE limitations:")
print("  - Penalize valid paraphrases ('needs' vs 'requires')")
print("  - Do not capture semantic correctness")
print("  - High BLEU is neither necessary nor sufficient for quality")


## LLM-as-Judge Design & Biases

| Bias | Description | Mitigation |
|---|---|---|
| **Position bias** | Model prefers the response shown first | Swap A/B order and average |
| **Verbosity bias** | Model prefers longer responses regardless of quality | Instruct judge to focus on content, not length |
| **Self-preference** | Model prefers its own outputs | Use a different model as judge |
| **Sycophancy** | Model agrees with evaluator's framing | Use neutral, double-blind prompts |


In [ ]:
# ===== Position Bias in LLM-as-Judge Demo =====
# Simulate judge responses showing position bias

def mock_llm_judge(response_a, response_b, seed=42):
    """
    Simulate a biased judge that prefers the FIRST response
    more than it should (position bias).
    """
    np.random.seed(seed)
    # Position bias: 60% chance to pick first regardless of quality
    if np.random.rand() < 0.6:
        return 'A'   # biased toward first
    # 40% chance to judge on content (random here)
    return np.random.choice(['A', 'B'])

def judge_with_position_control(response_a, response_b, n_evals=100):
    """
    Run judge in both orders (A,B) and (B,A) to cancel position bias.
    Final: A wins if it wins more than 50% of swapped evaluations.
    """
    wins_a = 0
    for i in range(n_evals // 2):
        # Order 1: A first
        if mock_llm_judge(response_a, response_b, seed=i) == 'A':
            wins_a += 1
        # Order 2: B first (judge sees B as "A")
        verdict = mock_llm_judge(response_b, response_a, seed=i + n_evals)
        if verdict == 'B':   # judge picked second = actually A
            wins_a += 1
    
    win_rate_a = wins_a / n_evals
    return win_rate_a

# Without position control
wins_naive = sum(1 for i in range(100) if mock_llm_judge("resp_A", "resp_B", seed=i) == 'A')
print(f"Naive judging: A wins {wins_naive}% of the time (biased by position)")

# With position control
win_rate = judge_with_position_control("resp_A", "resp_B", n_evals=100)
print(f"With order swapping: A wins {win_rate*100:.0f}% of the time (debiased)")
print()
print("If A and B are equal quality, we expect ~50% after debiasing.")


## Hallucination Measurement

**Grounding-based:** Check if each factual claim in the model output is supported by a retrieved document. Can be done with a separate NLI model or LLM checker.

**Model-based:** Ask a judge LLM: "Is the following claim supported by this document? [claim] [document]"

**Statistical:** Measure consistency of multiple sampled responses (SelfCheckGPT). High-probability facts get repeated; hallucinations vary across samples.


## Full Eval Stack

```
Automated (fast, cheap, runs on every PR)
  ├── BLEU/ROUGE (for translation, summarization)
  ├── Exact match / F1 on golden set (factual QA)
  ├── Task completion rate on eval suite
  └── Behavioral tests (format compliance, safety filters)

LLM-as-Judge (medium cost, runs nightly)
  ├── Pairwise comparison vs. baseline model
  ├── Rubric-based scoring (correctness, helpfulness, safety)
  └── Position-debiased via order swapping

Human Evaluation (slow, expensive, pre-launch)
  ├── Annotation of random sample (pairwise or rating)
  └── Expert review for domain-specific quality

Online Signals (free, ground-truth, delayed)
  ├── Thumbs up/down per response
  ├── Follow-up question rate (low = resolved)
  ├── Escalation/complaint rate
  └── Task completion (did user accomplish their goal?)
```


## Common Interview Questions

**Q: What are the key limitations of BLEU/ROUGE for LLM evaluation?**
They measure n-gram overlap, not semantic correctness. A valid paraphrase scores low if it doesn't share words with the reference. They don't capture factual accuracy at all — a fluent hallucination scores high if it uses the right words. BLEU was designed for translation (single correct answer); open-ended generation has many valid answers. Use them as sanity checks, not primary metrics.

**Q: How do you implement an LLM-as-judge and correct for position bias?**
Build a pairwise prompt: "Which response is better, A or B? Respond with exactly 'A' or 'B'." Run it twice — once with (A,B) order, once with (B,A). If the judge picks A first and B second (when B was shown first), A is genuinely better. Aggregate win rates across both orderings. This cancels position bias statistically.

**Q: What is SelfCheckGPT and when is it useful?**
SelfCheckGPT measures hallucination by sampling the same model multiple times and measuring agreement across responses. Factual statements the model "knows" appear consistently across samples; hallucinated content varies. This requires no external reference and works even without a golden answer — useful for evaluating factuality of open-ended generation.

**Q: What should a golden evaluation set contain?**
Diverse queries across the use case distribution (short/long, simple/complex, in-domain/edge-cases). High-quality reference answers written or verified by domain experts. Metadata: query type, difficulty, expected answer. Contamination checks — ensure test queries don't appear in training data. 200–500 examples is usually enough for offline tracking if well-curated.

## Key Takeaways
- BLEU/ROUGE measure n-gram overlap — useful sanity checks but miss semantics and factual correctness
- LLM-as-judge: pairwise with order swapping to cancel position bias; rubric-based for single rating
- Hallucination: grounding-based (does output cite retrieved docs?), consistency-based (SelfCheckGPT)
- Full eval stack: automated (PR) → LLM judge (nightly) → human eval (pre-launch) → online signals
- Golden set: 200–500 diverse, expert-verified, contamination-checked examples
- Online signals are ground truth — thumbs, follow-ups, escalation rate — but delayed; complement with offline